# Healthcare RCM Charge-Reimbursement Analytics

## Purpose
To analyze hospital-reported charges and estimated Medicare payments, calculate charge-to-payment metrics, and investigate records where Medicare payments exceed submitted charges.

## Dataset
- **Source:** CMS Medicare Inpatient Hospitals — by Provider and Service
- **File:** `data/MUP_INP_RY26_P03_V10_DY24_PrvSvc.csv`
- **Data year:** 2024

## 1. Dataset Loading

In this step, I load the CMS CSV dataset into a Pandas DataFrame. Before performing calculations, I will inspect the dataset's structure and check for potential data quality issues, including:

- Dataset dimensions (number of rows and columns)
- Missing values
- Duplicate records
- Inconsistent or invalid values

### Why Load Identifier Columns as Text?

Columns such as `Rndrng_Prvdr_CCN`, `Rndrng_Prvdr_Zip5`, `Rndrng_Prvdr_State_FIPS`, and `DRG_Cd` are identifiers, not numerical measurements.

I load these columns as text to preserve leading zeros. If they were interpreted as integers, leading zeros could be removed, potentially changing the original identifiers.

**Example:** Provider ID `010001` could become `10001`, which would no longer match the original identifier.

In [8]:
import pandas as pd

data_types = {
    "Rndrng_Prvdr_CCN": "str",
    "Rndrng_Prvdr_State_FIPS": "str",
    "Rndrng_Prvdr_Zip5": "str",
    "DRG_Cd": "str"
}
df = pd.read_csv('../data/MUP_INP_RY26_P03_V10_DY24_PrvSvc.csv', dtype=data_types)

In [9]:
print(df.shape)
print(df.columns)
print(df.dtypes)

(145879, 15)
Index(['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City',
       'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5',
       'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA',
       'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs',
       'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt'],
      dtype='str')
Rndrng_Prvdr_CCN                 str
Rndrng_Prvdr_Org_Name            str
Rndrng_Prvdr_City                str
Rndrng_Prvdr_St                  str
Rndrng_Prvdr_State_FIPS          str
Rndrng_Prvdr_Zip5                str
Rndrng_Prvdr_State_Abrvtn        str
Rndrng_Prvdr_RUCA            float64
Rndrng_Prvdr_RUCA_Desc           str
DRG_Cd                           str
DRG_Desc                         str
Tot_Dschrgs                    int64
Avg_Submtd_Cvrd_Chrg         float64
Avg_Tot_Pymt_Amt             float64
Avg_Mdcr_Pymt_Amt            float64
dtype: object


In [12]:
print(df.isna().sum())
print(f"Number of duplicate rows: {df.duplicated().sum()}")

Rndrng_Prvdr_CCN             0
Rndrng_Prvdr_Org_Name        0
Rndrng_Prvdr_City            0
Rndrng_Prvdr_St              0
Rndrng_Prvdr_State_FIPS      0
Rndrng_Prvdr_Zip5            0
Rndrng_Prvdr_State_Abrvtn    0
Rndrng_Prvdr_RUCA            0
Rndrng_Prvdr_RUCA_Desc       0
DRG_Cd                       0
DRG_Desc                     0
Tot_Dschrgs                  0
Avg_Submtd_Cvrd_Chrg         0
Avg_Tot_Pymt_Amt             0
Avg_Mdcr_Pymt_Amt            0
dtype: int64
Number of duplicate rows: 0


In [13]:
is_duplicate = df[["Rndrng_Prvdr_CCN", "DRG_Cd"]]
print(f"Number of duplicate Prvdr_DRG combinations: {is_duplicate.duplicated().sum()}")

Number of duplicate Prvdr_DRG combinations: 0


In [15]:
print(f"Number of unique Rndrng_Prvdr_CCN length: {df['Rndrng_Prvdr_CCN'].str.len().nunique()}")
print(f"Number of unique Rndrng_Prvdr_State_FIPS length: {df['Rndrng_Prvdr_State_FIPS'].str.len().nunique()}")
print(f"Number of unique Rndrng_Prvdr_Zip5 length: {df['Rndrng_Prvdr_Zip5'].str.len().nunique()}")
print(f"Number of unique DRG_Cd length: {df['DRG_Cd'].str.len().nunique()}")

Number of unique Rndrng_Prvdr_CCN length: 1
Number of unique Rndrng_Prvdr_State_FIPS length: 1
Number of unique Rndrng_Prvdr_Zip5 length: 1
Number of unique DRG_Cd length: 1


In [16]:
print(f"Min value of Tot_Dschrgs: {df['Tot_Dschrgs'].min()}, Max value of Tot_Dschrgs {df['Tot_Dschrgs'].max()}")
print(f"Min value of Avg_Submtd_Cvrd_Chrg: {df['Avg_Submtd_Cvrd_Chrg'].min()}, Max value of Avg_Submtd_Cvrd_Chrg: {df['Avg_Submtd_Cvrd_Chrg'].max()}")
print(f"Min value of Avg_Tot_Pymt_Amt: {df['Avg_Tot_Pymt_Amt'].min()}, Max value of Avg_Tot_Pymt_Amt: {df['Avg_Tot_Pymt_Amt'].max()}")
print(f"Min value of Avg_Mdcr_Pymt_Amt {df['Avg_Mdcr_Pymt_Amt'].min()}, Max value of Avg_Mdcr_Pymt_Amt: {df['Avg_Mdcr_Pymt_Amt'].max()}")

Min value of Tot_Dschrgs: 11, Max value of Tot_Dschrgs 3400
Min value of Avg_Submtd_Cvrd_Chrg: 2058.3846154, Max value of Avg_Submtd_Cvrd_Chrg: 7196636.9375
Min value of Avg_Tot_Pymt_Amt: 1849.0769231, Max value of Avg_Tot_Pymt_Amt: 1443309.6667
Min value of Avg_Mdcr_Pymt_Amt 386.8, Max value of Avg_Mdcr_Pymt_Amt: 1436667.8333


In [17]:
df["Payment_Exceeds_Charge"] = df["Avg_Submtd_Cvrd_Chrg"] < df["Avg_Mdcr_Pymt_Amt"]
print(df["Payment_Exceeds_Charge"].value_counts())

Payment_Exceeds_Charge
False    145758
True        121
Name: count, dtype: int64


## Initial Data Quality Findings

1. **Dataset size:** 145,879 rows and 15 columns.
2. **Missing values:** None found.
3. **Exact duplicate rows:** None found.
4. **Duplicate provider–DRG combinations:** None found.
5. **Identifier lengths:** Consistent across the four checked identifier columns.
6. **Numeric validation:** The four selected numeric columns contain no zero or negative values.
7. **Payment exceeds charge:** 121 records flagged for further investigation (approximately 0.083% of records).

**Note:** Flagged records are retained for investigation and are not assumed to be errors.

In [21]:
df["Total_Charges"] = df["Tot_Dschrgs"] * df["Avg_Submtd_Cvrd_Chrg"]
df["Total_Mdcr_Pymt"] = df["Tot_Dschrgs"] * df["Avg_Mdcr_Pymt_Amt"]
df["Chrg_to_Mdcr_Pymt_Ratio"] = df["Total_Charges"] / df["Total_Mdcr_Pymt"]


In [22]:
print(f"Overall Total Discharge: {df["Tot_Dschrgs"].sum()}")
print(f"Overall Total Charges: {df["Total_Charges"].sum()}")
print(f"Overall Total Medicare Payment: {df["Total_Mdcr_Pymt"].sum()}")

Overall Total Discharge: 4952481
Overall Total Charges: 457648118614.09705
Overall Total Medicare Payment: 75111479229.00195


In [24]:
print(f"Overall charge-to-payment ratio: {df["Total_Charges"].sum() / df["Total_Mdcr_Pymt"].sum()}")

Overall charge-to-payment ratio: 6.092918463485545
